In [ ]:
def main(datasources, start_date, end_date):
    """Return one PIT-safe tail-VWAP plus lagged trade-size fusion factor."""

    import numpy as np
    import pandas as pd
    import dai

    if isinstance(datasources, dict):
        datasource = next(
            (datasources[key] for key in ("bar1m", "stock_bar1m", "bigalpha_2026_stock_bar1m")
             if datasources.get(key)),
            None,
        )
    else:
        datasource = None
    datasource = datasource or "bigalpha_2026_stock_bar1m"
    start_ts = pd.Timestamp(start_date).normalize()
    end_ts = pd.Timestamp(end_date).normalize()
    query_start = start_ts - pd.DateOffset(days=20)

    def _daily_rank(values, dates):
        values = pd.to_numeric(values, errors="coerce")
        return values.groupby(dates, sort=False).rank(method="average", pct=True).fillna(0.5)

    def _daily_zscore(values, dates):
        values = pd.to_numeric(values, errors="coerce")
        grouped = values.groupby(dates, sort=False)
        mean = grouped.transform("mean")
        std = grouped.transform("std").replace(0, np.nan)
        return ((values - mean) / std).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    sql_template = """
        WITH daily AS (
            SELECT
                strftime(date, '%Y-%m-%d') AS trading_day,
                instrument,
                last(close ORDER BY date) AS last_close,
                SUM(CASE WHEN strftime(date, '%H:%M:%S') >= '14:30:00' THEN amount ELSE 0 END) AS tail_amount,
                SUM(CASE WHEN strftime(date, '%H:%M:%S') >= '14:30:00' THEN volume ELSE 0 END) AS tail_volume,
                SUM(amount) AS amount_sum,
                SUM(deal_number) AS deal_sum,
                SUM(CASE WHEN strftime(date, '%H:%M:%S') <= '10:30:00' THEN amount ELSE 0 END) AS early_amount,
                SUM(CASE WHEN strftime(date, '%H:%M:%S') <= '10:30:00' THEN deal_number ELSE 0 END) AS early_deal,
                SUM(CASE WHEN strftime(date, '%H:%M:%S') >= '14:00:00' THEN amount ELSE 0 END) AS late_amount,
                SUM(CASE WHEN strftime(date, '%H:%M:%S') >= '14:00:00' THEN deal_number ELSE 0 END) AS late_deal
            FROM {datasource}
            GROUP BY trading_day, instrument
        )
        SELECT
            CAST(trading_day AS DATETIME) AS date,
            instrument,
            last_close / NULLIF(tail_amount / NULLIF(tail_volume, 0), 0) - 1 AS tail_vwap_dev,
            amount_sum, deal_sum, early_amount, early_deal, late_amount, late_deal
        FROM daily
    """
    fallback_sql_template = sql_template.replace("deal_number", "volume")
    frame = None
    for template in (sql_template, fallback_sql_template):
        try:
            frame = dai.query(
                template.format(datasource=datasource),
                filters={"date": [query_start, end_ts + pd.DateOffset(days=1)]},
                compression=True,
            ).df()
            if frame is not None and not frame.empty:
                break
        except Exception:
            frame = None
    if frame is None or frame.empty:
        raise RuntimeError("NO_FACTOR_ROWS_FROM_BIGQUANT_DAI")

    frame = frame.copy()
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce").dt.normalize()
    frame["instrument"] = frame["instrument"].astype(str)
    frame = frame.dropna(subset=["date", "instrument"]).sort_values(["instrument", "date"])
    for column in ["tail_vwap_dev", "amount_sum", "deal_sum", "early_amount", "early_deal", "late_amount", "late_deal"]:
        frame[column] = pd.to_numeric(frame.get(column, 0.0), errors="coerce").replace([np.inf, -np.inf], np.nan)
    frame["avg_size"] = frame["amount_sum"] / frame["deal_sum"].where(frame["deal_sum"] > 0, np.nan)
    frame["early_size"] = frame["early_amount"] / frame["early_deal"].where(frame["early_deal"] > 0, np.nan)
    frame["late_size"] = frame["late_amount"] / frame["late_deal"].where(frame["late_deal"] > 0, np.nan)
    frame["late_share"] = frame["late_amount"] / frame["amount_sum"].where(frame["amount_sum"] > 0, np.nan)
    lag_columns = ["avg_size", "early_size", "late_size", "late_share"]
    frame[lag_columns] = frame.groupby("instrument", sort=False)[lag_columns].shift(1)
    frame = frame.loc[(frame["date"] >= start_ts) & (frame["date"] <= end_ts)].copy()
    if frame.empty:
        raise RuntimeError("NO_FACTOR_ROWS_IN_REQUESTED_RANGE")

    tail_raw = pd.to_numeric(frame["tail_vwap_dev"], errors="coerce").fillna(0.0)
    tail_factor = -1.0 * (_daily_rank(_daily_zscore(tail_raw, frame["date"]), frame["date"]) * np.sign(tail_raw))
    size_rank = _daily_rank(frame["avg_size"], frame["date"])
    early_rank = _daily_rank(frame["early_size"], frame["date"])
    late_rank = _daily_rank(frame["late_size"], frame["date"])
    share_rank = _daily_rank(frame["late_share"], frame["date"])
    lag_raw = (size_rank - 0.5) * (late_rank - early_rank) * (share_rank - 0.5)
    lag_factor = _daily_rank(lag_raw, frame["date"])
    amount_rank = _daily_rank(frame["amount_sum"], frame["date"])
    fused = tail_factor * (0.50 + 0.50 * (1.0 - lag_factor))
    frame["factor"] = _daily_rank(pd.to_numeric(fused, errors="coerce"), frame["date"]).fillna(0.5)
    out = frame[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)
    return out
